# 02b - Baseline GNN classifiers: Temporal GCN and GraphSAGE

Trains two graph neural networks on the Elliptic transaction graph.
Checkpoints are saved under `models/saved/`.


In [ ]:
import numpy as np
import pandas as pd
import torch

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.data import load_elliptic, preprocess_elliptic
from xai_blockchain_framework.models.gnn import (
    GraphSAGE, TemporalGCN, build_edge_index, get_device, train_gnn,
)
from xai_blockchain_framework.utils import save_csv

set_seed()
device = get_device()
print(f'Using device: {device}')


## Build the transaction graph

In [ ]:
raw = load_elliptic()
X, y, ts = preprocess_elliptic(raw, drop_unknown=False, normalize=True)

# Remap transaction ids to row indices for the edge index.
tx_to_idx = {tx: i for i, tx in enumerate(raw.tx_ids.tolist())}
edges = raw.edges.copy()
edges.iloc[:, 0] = edges.iloc[:, 0].map(tx_to_idx)
edges.iloc[:, 1] = edges.iloc[:, 1].map(tx_to_idx)
edges = edges.dropna().astype(int)
edge_index = build_edge_index(edges).to(device)

x_t  = torch.tensor(X, dtype=torch.float32, device=device)
y_t  = torch.tensor(np.clip(y, 0, None), dtype=torch.long, device=device)
ts_t = torch.tensor(ts, dtype=torch.long, device=device)

labeled = y != -1
train_mask = torch.tensor(labeled & (ts <= 34), dtype=torch.bool, device=device)
val_mask   = torch.tensor(labeled & (ts >= 35) & (ts <= 39), dtype=torch.bool, device=device)
test_mask  = torch.tensor(labeled & (ts >= 40), dtype=torch.bool, device=device)


## Train the two GNN models

In [ ]:
reports = []
for name, model_cls, kwargs in [
    ('TemporalGCN', TemporalGCN, {}),
    ('GraphSAGE',   GraphSAGE,   {}),
]:
    print(f'--- {name} ---')
    model = model_cls(in_features=X.shape[1], **kwargs).to(device)
    time_steps_arg = ts_t if name == 'TemporalGCN' else None
    result = train_gnn(
        model, x_t, edge_index, y_t,
        train_mask=train_mask, val_mask=val_mask,
        time_steps=time_steps_arg,
    )
    reports.append({'Model': name, 'BestEpoch': result.best_epoch, 'BestValAUC': result.best_val_auc})
    torch.save(model.state_dict(), PATHS.models_dir / f'elliptic_{name.lower()}.pt')

df = pd.DataFrame(reports)
print(df.to_string(index=False))
save_csv(df, PATHS.results_dir / 'gnn_baselines_results.csv')


## Save graph tensors for downstream notebooks

In [ ]:
torch.save(edge_index.cpu(), PATHS.data_processed / 'elliptic_edge_index.pt')
torch.save(x_t.cpu(),        PATHS.data_processed / 'elliptic_node_features.pt')
torch.save(y_t.cpu(),        PATHS.data_processed / 'elliptic_node_labels.pt')
torch.save(ts_t.cpu(),       PATHS.data_processed / 'elliptic_time_steps.pt')
print('Graph tensors saved to data/processed/')
